# Chapter 50: LLM Fine-Tuning for Kaggle

How does changing LoRA rank change adapter parameter count and the constructed matrix update?

I would use the chapter as an experiment menu rather than a recipe that must always be applied in full. The truncation examples help me make a meaningful data decision. LoRA’s explanation is less precise about scaling versus learning rate and frozen weights. A tiny matrix update lets me inspect the mechanics before accepting parameter-efficiency claims. Create a four-by-three frozen matrix and explicit rank-r factors A and B. Compute their scaled product B@A with alpha/r scaling, add it to the frozen matrix, and report the adapter parameter count, full matrix parameter count, and updated matrix entries. The control is rank. The count is r*(input_dimension+output_dimension). At small dimensions and large ranks it can exceed the full matrix count. Changing rank also changes alpha/r scaling, so a fair rank experiment should state which scaling convention is held fixed. Compare adapter rank one and three. State why the result counts only the update matrices rather than optimizer states, activations, or the frozen backbone, and list what a memory estimate still needs.

This is a constructed teaching example. Its results describe these supplied inputs, not a measured competition gain.

## Assumptions

Small dimensions deliberately reveal that adapters are not always smaller. Factors are deterministic illustrative numbers, not trained values. Alpha stays fixed at 2.

In [1]:
parameter = 1
# Change this input, then run the following calculation.

In [2]:
import json
assert int(parameter)==parameter and 1<=parameter<=3
r=int(parameter); ins=3; outs=4; alpha=2
A=[[(i+1)*(j+1)/10 for j in range(ins)] for i in range(r)]
B=[[(i+1)/(j+2) for j in range(r)] for i in range(outs)]
W=[[int(i==j) for j in range(ins)] for i in range(outs)]
update=[[alpha/r*sum(B[i][k]*A[k][j] for k in range(r)) for j in range(ins)] for i in range(outs)]
assert len(update)==outs and all(len(x)==ins for x in update)
print(json.dumps({'rank':r,'scale':alpha/r,'adapter_parameters':r*(ins+outs),'full_parameters':ins*outs,'table':[{'row':i,'updated_values':[W[i][j]+update[i][j] for j in range(ins)]} for i in range(outs)]}))


{"rank": 1, "scale": 2.0, "adapter_parameters": 7, "full_parameters": 12, "table": [{"row": 0, "updated_values": [1.1, 0.2, 0.3]}, {"row": 1, "updated_values": [0.2, 1.4, 0.6]}, {"row": 2, "updated_values": [0.30000000000000004, 0.6000000000000001, 1.9]}, {"row": 3, "updated_values": [0.4, 0.8, 1.2]}]}


## Interpretation

The count is r*(input_dimension+output_dimension). At small dimensions and large ranks it can exceed the full matrix count. Changing rank also changes alpha/r scaling, so a fair rank experiment should state which scaling convention is held fixed.

Book location: Chapter 50, ## Why LoRA Changes the Game.

[Technical corrections](https://github.com/KarpelesPublishing/winning-kaggle-companion/blob/main/docs/errata.md).